# 06. Aplicación y visualización Databricks desde refined

Consume tablas Databrickses y exporta figuras/CSV para el informe.


In [0]:
%run ./00_configuracion


In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TABLAS = {
    "conteo_clases": "refined_eda_conteo_clases",
    "desbalance_clases": "refined_eda_desbalance_clases",
    "expresion_global": "refined_eda_expresion_global",
    "genes_mas_variables": "refined_eda_genes_mas_variables",
    "importancias_rfe": "refined_rfe_feature_importances",
    "metricas_modelos": "refined_metricas_modelos_sparkml",
    "hiperparametros": "refined_hiperparametros_modelos_sparkml",
    "reporte_clase": "refined_reporte_clasificacion_por_clase",
    "confusion": "refined_matriz_confusion_mejor_modelo",
    "predicciones": "refined_predicciones_test_mejor_modelo",
}

def cargar(alias, obligatoria=True):
    nombre = TABLAS[alias]
    if table_exists_databricks(nombre):
        print("Tabla encontrada:", nombre)
        return load_spark_table(nombre)
    if obligatoria:
        raise FileNotFoundError(f"No se encontró {nombre}. Ejecute los notebooks anteriores.")
    print("Advertencia: no existe", nombre)
    return None

df_clases = cargar("conteo_clases")
df_desbalance = cargar("desbalance_clases")
df_metricas = cargar("metricas_modelos")
df_reporte = cargar("reporte_clase")
df_confusion = cargar("confusion")
df_pred = cargar("predicciones")

df_expr = cargar("expresion_global", False)
df_genes_var = cargar("genes_mas_variables", False)
df_importancias_rfe = cargar("importancias_rfe", False)
df_hiper = cargar("hiperparametros", False)


In [0]:
n_clases = df_clases.select("cancer_type").distinct().count()
n_muestras = df_clases.agg(F.sum("n_muestras").alias("n")).collect()[0]["n"]

mejor_val = (
    df_metricas
    .filter(F.col("split") == "validation")
    .orderBy(F.desc("f1_macro"), F.desc("balanced_accuracy"))
    .limit(1)
    .collect()[0]
)
modelo_final = mejor_val["modelo"]

test_row = (
    df_metricas
    .filter((F.col("split") == "test") & (F.col("modelo") == modelo_final))
    .limit(1)
    .collect()
)
test_row = test_row[0] if test_row else None

resumen = pd.DataFrame([{
    "n_clases": int(n_clases),
    "n_muestras": int(n_muestras),
    "modelo_final": modelo_final,
    "f1_macro_validation": float(mejor_val["f1_macro"]),
    "balanced_accuracy_validation": float(mejor_val["balanced_accuracy"]),
    "f1_macro_test": float(test_row["f1_macro"]) if test_row else None,
    "balanced_accuracy_test": float(test_row["balanced_accuracy"]) if test_row else None,
}])
display(resumen)
save_spark_table(spark.createDataFrame(resumen), "refined_app_resumen_ejecutivo")


In [0]:
pdf_clases = df_clases.toPandas().sort_values("n_muestras", ascending=False)
mostrar(df_clases.orderBy(F.desc("n_muestras")), 30)

plt.figure(figsize=(13,5))
plt.bar(pdf_clases["cancer_type"], pdf_clases["n_muestras"])
plt.title("Distribución de muestras por tipo de cáncer")
plt.xlabel("Tipo de cáncer")
plt.ylabel("Número de muestras")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
guardar_figura("app_01_distribucion_clases.png")
plt.show()

pdf_desbalance = df_desbalance.toPandas().sort_values("n_muestras", ascending=False)
if "porcentaje" in pdf_desbalance.columns:
    plt.figure(figsize=(13,5))
    plt.bar(pdf_desbalance["cancer_type"], pdf_desbalance["porcentaje"])
    plt.title("Participación porcentual de cada clase")
    plt.xlabel("Tipo de cáncer")
    plt.ylabel("Porcentaje (%)")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura("app_02_porcentaje_clases.png")
    plt.show()


In [0]:
columnas_metricas = [c for c in [
    "modelo", "split", "accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_macro_ovr", "pr_auc_macro_ovr",
    "gap_f1_macro_train_validation", "gap_accuracy_train_validation"
] if c in df_metricas.columns]

tabla_metricas = df_metricas.select(*columnas_metricas).orderBy("split", F.desc("f1_macro"))
mostrar(tabla_metricas, 100)
pdf_metricas = tabla_metricas.toPandas()

def plot_metric(metric_col, title, filename):
    data = pdf_metricas[pdf_metricas["split"].isin(["validation", "test"])]
    if data.empty or metric_col not in data.columns:
        return
    pivot = data.pivot_table(index="modelo", columns="split", values=metric_col, aggfunc="first").sort_values(by="validation", ascending=False)
    ax = pivot.plot(kind="bar", figsize=(11,5), rot=30)
    ax.set_title(title)
    ax.set_xlabel("Modelo")
    ax.set_ylabel(metric_col)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura(filename)
    plt.show()

plot_metric("f1_macro", "Comparación de modelos por F1 macro", "app_03_comparacion_modelos_f1_macro.png")
plot_metric("balanced_accuracy", "Comparación de modelos por balanced accuracy", "app_04_comparacion_modelos_balanced_accuracy.png")
plot_metric("pr_auc_macro_ovr", "Comparación de modelos por PR-AUC macro", "app_05_comparacion_modelos_pr_auc_macro.png")
plot_metric("roc_auc_macro_ovr", "Comparación de modelos por ROC-AUC macro", "app_06_comparacion_modelos_roc_auc_macro.png")

if df_hiper is not None:
    mostrar(df_hiper.orderBy("modelo", "parametro"), 100)


In [0]:
pdf_reporte = df_reporte.toPandas().copy()
filas_agregadas = {"accuracy", "macro avg", "weighted avg"}
pdf_reporte_clases = pdf_reporte[~pdf_reporte["clase"].isin(filas_agregadas)].copy() if "clase" in pdf_reporte.columns else pdf_reporte.copy()

for col in ["precision", "recall", "f1-score", "support"]:
    if col in pdf_reporte_clases.columns:
        pdf_reporte_clases[col] = pd.to_numeric(pdf_reporte_clases[col], errors="coerce")

if "f1-score" in pdf_reporte_clases.columns:
    pdf_reporte_clases = pdf_reporte_clases.sort_values("f1-score", ascending=False)

display(pdf_reporte_clases)
save_spark_table(spark.createDataFrame(pdf_reporte_clases), "refined_app_desempeno_por_clase")

if {"clase", "f1-score"}.issubset(pdf_reporte_clases.columns):
    plt.figure(figsize=(13,5))
    plt.bar(pdf_reporte_clases["clase"], pdf_reporte_clases["f1-score"])
    plt.title("F1-score por clase en test — mejor modelo")
    plt.xlabel("Tipo de cáncer")
    plt.ylabel("F1-score")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura("app_07_f1_por_clase_test.png")
    plt.show()


In [0]:
pdf_confusion = df_confusion.toPandas()
tabla_confusion = pdf_confusion.pivot_table(index="true_class", columns="predicted_class", values="n", aggfunc="sum", fill_value=0)
clases = list(tabla_confusion.index)
tabla_confusion = tabla_confusion.reindex(index=clases, columns=clases, fill_value=0)
display(tabla_confusion)

plt.figure(figsize=(11,9))
plt.imshow(tabla_confusion.values, interpolation="nearest")
plt.title("Matriz de confusión — conteos absolutos")
plt.colorbar(label="Número de muestras")
plt.xticks(np.arange(len(tabla_confusion.columns)), tabla_confusion.columns, rotation=90)
plt.yticks(np.arange(len(tabla_confusion.index)), tabla_confusion.index)
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
for i in range(tabla_confusion.shape[0]):
    for j in range(tabla_confusion.shape[1]):
        v = int(tabla_confusion.iloc[i, j])
        if v > 0:
            plt.text(j, i, str(v), ha="center", va="center", fontsize=7)
plt.tight_layout()
guardar_figura("app_09_matriz_confusion_absoluta.png")
plt.show()

tabla_norm = tabla_confusion.div(tabla_confusion.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
plt.figure(figsize=(11,9))
plt.imshow(tabla_norm.values, interpolation="nearest", vmin=0, vmax=1)
plt.title("Matriz de confusión normalizada por clase real")
plt.colorbar(label="Proporción")
plt.xticks(np.arange(len(tabla_norm.columns)), tabla_norm.columns, rotation=90)
plt.yticks(np.arange(len(tabla_norm.index)), tabla_norm.index)
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
plt.tight_layout()
guardar_figura("app_10_matriz_confusion_normalizada.png")
plt.show()


In [0]:
errores = (
    df_pred
    .withColumn("es_error", F.when(F.col("cancer_type") != F.col("cancer_predicho"), F.lit(1)).otherwise(F.lit(0)))
    .filter(F.col("es_error") == 1)
    .groupBy("cancer_type", "cancer_predicho")
    .agg(F.count("*").alias("n_errores"))
    .orderBy(F.desc("n_errores"))
)
mostrar(errores, 50)
save_spark_table(errores, "refined_app_errores_frecuentes")

pdf_errores = errores.limit(15).toPandas()
if not pdf_errores.empty:
    pdf_errores["par_error"] = pdf_errores["cancer_type"] + " → " + pdf_errores["cancer_predicho"]
    plt.figure(figsize=(12,5))
    plt.bar(pdf_errores["par_error"], pdf_errores["n_errores"])
    plt.title("Top 15 errores más frecuentes")
    plt.xlabel("Clase real → clase predicha")
    plt.ylabel("Número de errores")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura("app_11_top_errores_frecuentes.png")
    plt.show()

auditoria = (
    df_pred
    .withColumn("acierto", F.when(F.col("cancer_type") == F.col("cancer_predicho"), F.lit(1)).otherwise(F.lit(0)))
    .groupBy("cancer_type")
    .agg(F.count("*").alias("n_test"), F.sum("acierto").alias("n_aciertos"), F.round(F.avg("acierto"), 4).alias("accuracy_por_clase"))
    .orderBy(F.asc("accuracy_por_clase"), F.desc("n_test"))
)
mostrar(auditoria, 30)
save_spark_table(auditoria, "refined_app_auditoria_predicciones")


In [0]:
if df_importancias_rfe is not None:
    pdf_imp = df_importancias_rfe.orderBy(F.desc("importance")).limit(20).toPandas()
    display(pdf_imp)
    gene_col = "gene" if "gene" in pdf_imp.columns else "gene_id_base"
    plt.figure(figsize=(13,5))
    plt.bar(pdf_imp[gene_col], pdf_imp["importance"])
    plt.title("Top 20 genes por importancia RFE")
    plt.xlabel("Gen")
    plt.ylabel("Importancia")
    plt.xticks(rotation=60, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura("app_13_top20_genes_importancia_rfe.png")
    plt.show()

if df_expr is not None:
    pdf_expr = df_expr.toPandas().sort_values("media_log2_tpm", ascending=False)
    plt.figure(figsize=(13,5))
    plt.bar(pdf_expr["cancer_type"], pdf_expr["media_log2_tpm"])
    plt.title("Expresión promedio log2(TPM + 1) por tipo de cáncer")
    plt.xlabel("Tipo de cáncer")
    plt.ylabel("Media log2(TPM + 1)")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    guardar_figura("app_15_expresion_promedio_por_clase.png")
    plt.show()


In [0]:
# Exportación final a CSV
tabla_metricas.toPandas().to_csv(REFINED_VISUALIZATIONS_PATH / "app_tabla_metricas_informe.csv", index=False)
resumen.to_csv(REFINED_VISUALIZATIONS_PATH / "app_resumen_ejecutivo.csv", index=False)
pdf_reporte_clases.to_csv(REFINED_VISUALIZATIONS_PATH / "app_desempeno_por_clase.csv", index=False)
errores.toPandas().to_csv(REFINED_VISUALIZATIONS_PATH / "app_errores_frecuentes.csv", index=False)
auditoria.toPandas().to_csv(REFINED_VISUALIZATIONS_PATH / "app_auditoria_predicciones.csv", index=False)

print("CSV exportados en:", REFINED_VISUALIZATIONS_PATH)
